In [1]:
import os
from langchain.docstore.document import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor, tool
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
#!pip install -U duckduckgo-search

In [6]:


# 환경 변수 설정 (필요에 따라 주석 해제 후 API 키 입력)
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# 1. 영화 정보가 담긴 문서 리스트 생성
docs = [
    Document(
        page_content="크리스토퍼 놀란 감독의 SF 영화. 꿈속으로 들어가 현실을 조작한다.",
        metadata={
            "title": "인셉션",
            "director": "크리스토퍼 놀란",
            "year": 2010,
            "genre": "SF",
            "rating": 8.8,
        },
    ),
    Document(
        page_content="거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.",
        metadata={
            "title": "인터스텔라",
            "director": "크리스토퍼 놀란",
            "year": 2014,
            "genre": "SF",
            "rating": 8.6,
        },
    ),
    Document(
        page_content="미국 대공황 시대를 배경으로 한 마술 대결 영화.",
        metadata={
            "title": "프레스티지",
            "director": "크리스토퍼 놀란",
            "year": 2006,
            "genre": "미스터리",
            "rating": 8.5,
        },
    ),
    Document(
        page_content="스파이더맨의 능력과 책임에 대한 이야기. 고등학생 영웅의 성장기.",
        metadata={
            "title": "스파이더맨: 홈커밍",
            "director": "존 왓츠",
            "year": 2017,
            "genre": "액션",
            "rating": 7.4,
        },
    ),
    Document(
        page_content="마블 히어로들이 팀을 이루어 지구를 구하는 이야기. 화려한 액션이 특징.",
        metadata={
            "title": "어벤져스",
            "director": "조스 웨던",
            "year": 2012,
            "genre": "액션",
            "rating": 8.1,
        },
    ),
]

# 2. 벡터 스토어 및 임베딩 모델 설정
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(docs, embeddings)
llm = ChatOpenAI(temperature=0)

# 3. 셀프 쿼리 리트리버를 위한 메타데이터 스키마 정의
metadata_field_info = [
    AttributeInfo(name="title", description="영화의 제목", type="string"),
    AttributeInfo(name="director", description="영화 감독의 이름", type="string"),
    AttributeInfo(name="year", description="영화가 개봉된 연도", type="integer"),
    AttributeInfo(name="genre", description="영화의 장르", type="string"),
    AttributeInfo(name="rating", description="IMDb 평점 (10점 만점)", type="float"),
]

# 4. 셀프 쿼리 리트리버 생성
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="영화의 줄거리와 특징에 대한 설명입니다.",
    metadata_field_info=metadata_field_info,
    verbose=True,
)


# 5. 셀프 쿼리 리트리버를 툴로 정의
@tool
def movie_retriever_tool(query: str) -> str:
    """
    사용자의 자연어 쿼리를 기반으로 영화를 검색합니다.
    감독, 개봉 연도, 장르, 평점과 같은 조건을 함께 사용하여 검색할 수 있습니다.
    예시: "크리스토퍼 놀란 감독이 2010년 이후에 만든 SF 영화"
    """
    return self_query_retriever.invoke(query)


# 6. 두 번째 툴: 인터넷 검색
internet_search = DuckDuckGoSearchRun()


@tool
def internet_search_tool(query: str) -> str:
    """
    웹에서 최신 정보를 검색하는 도구입니다. 영화 정보를 찾을 때 사용하세요.
    """
    try:
        return internet_search.invoke(query)
    except Exception as e:
        return f"인터넷 검색 중 오류 발생: {str(e)}"


# 7. 에이전트 및 에이전트 실행자 생성
tools = [movie_retriever_tool, internet_search_tool]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 영화 검색 전문가입니다. 사용자 질문에 답변하기 위해 주어진 툴을 활용하세요.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# 8. 에이전트 실행
print("--- case 1: 복잡한 조건 검색 ---")
# "2010년 이후에 개봉한 크리스토퍼 놀란 감독의 SF 영화를 찾아줘"
result1 = agent_executor.invoke(
    {"input": "2010년 이후에 개봉한 크리스토퍼 놀란 감독의 SF 영화 알려줘"}
)
print(result1["output"])

--- case 1: 복잡한 조건 검색 ---


> Entering new AgentExecutor chain...

Invoking: `movie_retriever_tool` with `{'query': '크리스토퍼 놀란 감독이 2010년 이후에 만든 SF 영화'}`


[Document(id='8ed6237d-1e35-4833-bbf1-a8c9fb4207c5', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='230f8df8-788f-4e8e-8c76-924a43c675b8', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='fd413176-d88e-40b1-b97e-7604927ab0b7', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='20944f0e-4a3a-471c-b2c7-a92304e7acab', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.')]크리스토퍼 놀란 감독의 2010년 이후 SF 영

In [7]:
print("\n--- case 2: 단순한 쿼리 검색 ---")
# "마블 영화 중에서 평점 8.0점 이상인 영화를 찾아줘"
result2 = agent_executor.invoke(
    {"input": "마블 영화 중에서 평점 8.0점 이상인 영화를 찾아줘"}
)
print(result2["output"])


--- case 2: 단순한 쿼리 검색 ---


> Entering new AgentExecutor chain...

Invoking: `movie_retriever_tool` with `{'query': '마블 영화 중에서 평점 8.0점 이상인 영화'}`


[Document(id='90f045f4-60d0-427e-9a7c-13a7149b4cda', metadata={'genre': 'SF', 'title': '인터스텔라', 'rating': 8.6, 'director': '크리스토퍼 놀란', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='c44c50a1-610b-4042-b86c-1867fe7ed2fc', metadata={'rating': 8.6, 'genre': 'SF', 'director': '크리스토퍼 놀란', 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='852a6723-5975-4f67-9950-35b20134d698', metadata={'rating': 8.1, 'title': '어벤져스', 'genre': '액션', 'director': '조스 웨던', 'year': 2012}, page_content='마블 히어로들이 팀을 이루어 지구를 구하는 이야기. 화려한 액션이 특징.'), Document(id='5fd44573-db11-487f-870d-1dfbb91cb4ca', metadata={'year': 2006, 'rating': 8.5, 'title': '프레스티지', 'director': '크리스토퍼 놀란', 'genre': '미스터리'}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.')]마블 영화 중에서 평점 8.0점 이상인 영화를 찾았습니다:

1. **인터스텔

In [8]:
print("\n--- case 2: 단순한 쿼리 검색 ---")
# "마블 영화 중에서 평점 8.0점 이상인 영화를 찾아줘"
result2 = agent_executor.invoke({"input": "아이언맨1편에 대해 알려줘"})
print(result2["output"])


--- case 2: 단순한 쿼리 검색 ---


> Entering new AgentExecutor chain...

Invoking: `movie_retriever_tool` with `{'query': '아이언맨 1편'}`


[Document(id='5fd44573-db11-487f-870d-1dfbb91cb4ca', metadata={'genre': '미스터리', 'title': '프레스티지', 'year': 2006, 'director': '크리스토퍼 놀란', 'rating': 8.5}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.'), Document(id='2817d810-6867-408c-87de-ae88a5c3715e', metadata={'director': '크리스토퍼 놀란', 'year': 2006, 'genre': '미스터리', 'rating': 8.5, 'title': '프레스티지'}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.'), Document(id='0faddef2-970e-4bc9-9c7f-0a1aea30c935', metadata={'title': '스파이더맨: 홈커밍', 'year': 2017, 'rating': 7.4, 'director': '존 왓츠', 'genre': '액션'}, page_content='스파이더맨의 능력과 책임에 대한 이야기. 고등학생 영웅의 성장기.'), Document(id='913e6bc6-8ea3-49e6-a9d6-67775aad27e6', metadata={'title': '스파이더맨: 홈커밍', 'rating': 7.4, 'year': 2017, 'director': '존 왓츠', 'genre': '액션'}, page_content='스파이더맨의 능력과 책임에 대한 이야기. 고등학생 영웅의 성장기.')]
Invoking: `internet_search_tool` with `{'query': '아이언맨 1편 영화 정보'}`



c:\Users\SBA\AppData\Local\pypoetry\Cache\virtualenvs\langchain-kr-Us6BDj1P-py3.11\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
c:\Users\SBA\AppData\Local\pypoetry\Cache\virtualenvs\langchain-kr-Us6BDj1P-py3.11\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Jul 20, 2025 · 아이언이네?" 라고 한 말에 그대로 꽂혀 '아이언'으로 결정했다고 한다. 또 본명이 정 현 철인 서태지 와 혼동되는 경우도 가끔 있었다. 넉넉하지 않은 환경 때문에 광주, 익산, 창원, 서울, 목포 등지를 오가며 컸다. Jul 7, 2025 · 수많은 골프 아이언, 도대체 어떤 걸 사야 할까요? 이 글 하나로 복잡한 아이언의 세계를 완벽하게 정리해 드립니다. Apr 7, 2025 · 이번 포스팅에서는 중상급자부터 상급자 남성 골퍼들이 실제로 선택하고 있는 최고의 아이언 TOP 5 를 소개합니다. 각 브랜드의 철학, 기술력, 그리고 실제 투어 프로들이 사용하는 모델까지 꼼꼼히 분석해 드릴게요. 6 days ago · 단조 아이언 이면서도 초보자도 쉽게 접할 수 있으며 상급자 에게는 최고의 샷 메이킹을 위한 최적의 아이언 이라 생각 됩니다. Mar 25, 2025 · Originally posted on MFSgolf.com 2019.10.25 아이언 헤드의 선택 시 우리는 여러 가지 요소를 고려하여야 한다. 이때 고려되는 요소로는 첫째 단조 (forged) 헤드냐 주조 (casting) 헤드냐의 제작 방식에 대한 차이와 …아이언맨 1편에 대한 정보를 찾아보았습니다. 

1. **영화 정보:**
   - 제목: 아이언맨 (Iron Man)
   - 개봉 연도: 2008년
   - 감독: 존 파브로
   - 장르: 액션, SF
   - 평점: 7.9/10

2. **줄거리:**
   - "아이언맨"은 천재 무기 제조업자 토니 스타크가 자신의 기술을 이용하여 강력한 갑옷을 만들어 슈퍼히어로로 변신하는 이야기를 다룹니다. 토니 스타크는 자신의 기술이 악용되는 것을 막기 위해 아이언맨으로서 세계의 안전을 지키는 모험을 시작합니다.

이렇게 아이언맨 1편은 2008년에 개봉한 액션과 SF 장르의 영화로, 토니 스타크가 아이언맨으로 변신하여 세계의 안전을 위해 싸우는 이야기를 담고 있습니다.

> Finished chain.
아이언맨 1편에 대한 